# Visualization

## Data

### Datapath

In [1]:
import os

def get_game_layouts_directory(experiment:str):
    game_layouts = ["RANDOM", "SQUARE", "CIRCLE", "SIN", "GRID"] #* hardcoded for custom_health_gathering
    game_layouts_directory = {}
    for game_layout in game_layouts:
        game_layouts_directory[game_layout] = os.path.join(
            os.path.expanduser("~"), 
                f"Documents/Curious-Agent/train_dir/{experiment}/agent_trajectories/{game_layout}",
        )
    return game_layouts_directory

### Load Data

In [2]:
import torch

def load_episode(game_layouts_directory, game_layout: str, game_number:str):
    if game_layout not in game_layouts_directory:
        print(f"GAME_LAYOUT IS {game_layout} AND IS NOT IN {game_layouts_directory.keys()}")
        return [], []
    filepath = os.path.join(game_layouts_directory[game_layout], f"{game_number}.pt")
    if not os.path.exists(filepath):
        print(f"THE FILE {game_number}.pt DOES NOT EXIST IN {game_layouts_directory[game_layout]}")
        return [], []
    data = torch.load(filepath, map_location="cpu", weights_only=False)
    return data["frames"].numpy(), data["rewards"].squeeze(-1).numpy(), data["infos"]

In [3]:
import plotly.express as px
import pandas as pd

def show_rewards(rewards):
    df = pd.DataFrame({
        "Timestep": range(rewards.shape[0]),
        "Reward": rewards,
    })

    fig = px.line(
        df,
        x="Timestep",
        y="Reward",
        markers=True,
        title="Reward over time",
    )

    fig.show()

In [4]:
import cv2
import math
import numpy as np
import plotly.graph_objects as go

def show_episode(obs, rewards, frame_duration=200, scale:float=1):
    T = len(obs)

    if scale > 1:
        obs = np.stack([
                cv2.resize(
                    frame,
                    (math.ceil(frame.shape[1] * scale), math.ceil(frame.shape[0] * scale)),
                    interpolation=cv2.INTER_LANCZOS4,
                )
                for frame in obs
            ])

    frames = [
        go.Frame(
            data=[
                go.Image(z=obs[t])
            ],
            name=str(t),
            layout=go.Layout(
                title=f"Step = {t} | Reward = {rewards[t]:.4f}"
            ),
        )
        for t in range(T)
    ]

    fig = go.Figure(
        data=[
            go.Image(z=obs[0])
        ],
        frames=frames,
    )

    fig.update_layout(
        title=f"Step = 0 | Reward = {rewards[0]:.4f}",
        width=obs.shape[2] * 4,
        height=obs.shape[1] * 4,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        updatemenus=[
            {
                "type": "buttons",
                "buttons": [
                    {
                        "label": "▶ Play",
                        "method": "animate",
                        "args": [
                            None,
                            {
                                "frame": {
                                    "duration": frame_duration,
                                    "redraw": True,
                                },
                                "fromcurrent": True,
                            },
                        ],
                    },
                    {
                        "label": "⏸ Pause",
                        "method": "animate",
                        "args": [
                            [None],
                            {
                                "mode": "immediate",
                                "frame": {"duration": 0},
                                "transition": {"duration": 0},
                            },
                        ],
                    },
                ],
            }
        ],
        sliders=[
            {
                "steps": [
                    {
                        "method": "animate",
                        "label": str(t),
                        "args": [
                            [str(t)],
                            {
                                "mode": "immediate",
                                "frame": {"duration": 0, "redraw": True},
                                "transition": {"duration": 0},
                            },
                        ],
                    }
                    for t in range(T)
                ]
            }
        ],
    )

    fig.show()

In [8]:
import cv2
import math
import numpy as np
import base64
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def show_episode_with_rewards(obs, rewards, frame_duration=200, scale:float=1):
    T = len(obs)

    # 1. Scale images if needed
    if scale > 1:
        obs = np.stack([
            cv2.resize(
                frame,
                (math.ceil(frame.shape[1] * scale), math.ceil(frame.shape[0] * scale)),
                interpolation=cv2.INTER_LANCZOS4,
            )
            for frame in obs
        ])
        
    obs_height, obs_width = obs.shape[1], obs.shape[2]

    # 2. Pre-encode frames to base64 URIs
    img_sources = []
    for frame in obs:
        bgr_frame = frame[..., ::-1] 
        _, buffer = cv2.imencode('.jpg', bgr_frame)
        b64_str = base64.b64encode(buffer).decode('utf-8')
        img_sources.append(f"data:image/jpeg;base64,{b64_str}")

    # 3. Create subplots
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=("Environment", "Reward over Time"),
        column_widths=[0.5, 0.5]
    )

    # Trace 0: INVISIBLE scatter plot purely to set coordinate boundaries for the layout image
    fig.add_trace(
        go.Scatter(
            x=[0, obs_width], 
            y=[0, obs_height], 
            mode="markers", 
            marker=dict(color="rgba(0,0,0,0)"), 
            hoverinfo="none",
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Trace 1: Reward scatter plot
    fig.add_trace(
        go.Scatter(x=[0], y=[rewards[0]], mode='lines', name='Reward'),
        row=1, col=2
    )

    # 4. Build frames using layout.images instead of a data trace
    frames = [
        go.Frame(
            data=[
                # Pad the array with None to avoid DOM rebuilds on every tick
                go.Scatter(
                    x=list(range(T)), 
                    y=list(rewards[:t + 1]) + [None] * (T - t - 1), 
                    mode='lines'
                )
            ],
            name=str(t),
            traces=[1], # CRITICAL: Only update Trace 1 (Scatter). Ignore Trace 0.
            layout=go.Layout(
                title=f"Step = {t} | Reward = {rewards[t]:.4f}",
                images=[
                    dict(
                        source=img_sources[t],
                        xref="x1", yref="y1",
                        x=0, y=0,
                        sizex=obs_width, sizey=obs_height,
                        sizing="stretch",
                        layer="below"
                    )
                ]
            ),
        )
        for t in range(T)
    ]
    fig.frames = frames

    # 5. Configure Layout
    y_min, y_max = min(rewards), max(rewards)
    y_padding = (y_max - y_min) * 0.1 if y_max != y_min else 1.0

    fig.update_layout(
        title=f"Step = 0 | Reward = {rewards[0]:.4f}",
        width=obs_width + 600, 
        height=max(obs_height, 400),
        images=[
            dict(
                source=img_sources[0],
                xref="x1", yref="y1",
                x=0, y=0,
                sizex=obs_width, sizey=obs_height,
                sizing="stretch",
                layer="below"
            )
        ],
        updatemenus=[
            {
                "type": "buttons",
                "buttons": [
                    {
                        "label": "▶ Play",
                        "method": "animate",
                        "args": [
                            None,
                            {
                                "frame": {"duration": frame_duration, "redraw": True},
                                "fromcurrent": True,
                                "transition": {"duration": 0}, 
                            },
                        ],
                    },
                    {
                        "label": "⏸ Pause",
                        "method": "animate",
                        "args": [
                            [None],
                            {
                                "mode": "immediate",
                                "frame": {"duration": 0},
                                "transition": {"duration": 0},
                            },
                        ],
                    },
                ],
            }
        ],
        sliders=[
            {
                "steps": [
                    {
                        "method": "animate",
                        "label": str(t),
                        "args": [
                            [str(t)],
                            {
                                "mode": "immediate",
                                "frame": {"duration": 0, "redraw": True},
                                "transition": {"duration": 0},
                            },
                        ],
                    }
                    for t in range(T)
                ]
            }
        ],
    )

    # Reversing the y-axis range is necessary so the image renders top-to-bottom like a numpy array
    fig.update_xaxes(range=[0, obs_width], visible=False, row=1, col=1)
    fig.update_yaxes(range=[obs_height, 0], visible=False, row=1, col=1) 
    
    # Fix axes for the reward subplot
    fig.update_xaxes(title_text="Step", range=[0, T - 1], row=1, col=2)
    fig.update_yaxes(title_text="Reward", range=[y_min - y_padding, y_max + y_padding], row=1, col=2)

    fig.show()

# Loading Data

In [5]:
experiment = "rnd_reward_eval/visualize_/01_visualize_see_42_d.spe_10"
game_layouts_directory = get_game_layouts_directory(experiment)
print(game_layouts_directory)

obs, rewards, infos = load_episode(game_layouts_directory, "RANDOM", "0003512320")
print(f"OBS shape = {obs.shape}")
print(f"REWARDS shape = {rewards.shape}")

{'RANDOM': '/home/pinto/Documents/Curious-Agent/train_dir/rnd_reward_eval/visualize_/01_visualize_see_42_d.spe_10/agent_trajectories/RANDOM', 'SQUARE': '/home/pinto/Documents/Curious-Agent/train_dir/rnd_reward_eval/visualize_/01_visualize_see_42_d.spe_10/agent_trajectories/SQUARE', 'CIRCLE': '/home/pinto/Documents/Curious-Agent/train_dir/rnd_reward_eval/visualize_/01_visualize_see_42_d.spe_10/agent_trajectories/CIRCLE', 'SIN': '/home/pinto/Documents/Curious-Agent/train_dir/rnd_reward_eval/visualize_/01_visualize_see_42_d.spe_10/agent_trajectories/SIN', 'GRID': '/home/pinto/Documents/Curious-Agent/train_dir/rnd_reward_eval/visualize_/01_visualize_see_42_d.spe_10/agent_trajectories/GRID'}
OBS shape = (2100, 84, 168, 3)
REWARDS shape = (2100,)


In [6]:
show_rewards(rewards)

In [2]:
#show_episode(obs, rewards, 50, 1.5)

In [1]:
#show_episode_with_rewards(obs, rewards, 50, 1.5)